In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

In [2]:

%run data_extraction_feature_engineering.ipynb
df_train_full, df_test_full = extract_data()
VAL_START_DATE = '2012-09-01'
df_with_features = create_features1(df_train_full)
X_train, y_train, X_val, y_val = train_and_val_split(df_with_features, VAL_START_DATE, skip_dummies=False)


Loading data...
Train shape: (421570, 5)
Features shape: (8190, 12)
Stores shape: (45, 3)
Test shape: (115064, 4)
Merging datasets...

Data Preparation Complete!
X_train shape: (397841, 24)
X_val shape: (23729, 24)
   IsHoliday  Temperature  Fuel_Price  MarkDown1  MarkDown2  MarkDown3  \
0      False        42.31       2.572        0.0        0.0        0.0   
1       True        38.51       2.548        0.0        0.0        0.0   
2      False        39.93       2.514        0.0        0.0        0.0   
3      False        46.63       2.561        0.0        0.0        0.0   
4      False        46.50       2.625        0.0        0.0        0.0   

   MarkDown4  MarkDown5         CPI  Unemployment  ...  sales_lag_52  \
0        0.0        0.0  211.096358         8.106  ...           NaN   
1        0.0        0.0  211.242170         8.106  ...           NaN   
2        0.0        0.0  211.289143         8.106  ...           NaN   
3        0.0        0.0  211.319643         8.106  .

In [3]:
!pip install wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/sandro/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
!pip install xgboost


In [4]:
print(f"Holiday Weeks in Train: {X_train['IsHoliday'].sum()}")
print(f"Holiday Weeks in Val:   {X_val['IsHoliday'].sum()}")

Holiday Weeks in Train: 26695
Holiday Weeks in Val:   2966


In [5]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
import wandb
import os
import joblib

run = wandb.init(
    project="walmart-sales-forecasting",
    name="XGBoost_Baseline1",
    job_type="XGBoost",
    config={
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 500,
    'early_stopping_rounds': 20,
    'seed': 42,
    'verbosity': 1 
    }
)
# ---------------------------------------------------------
# TRAIN XGBOOST MODEL
# ---------------------------------------------------------

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'learning_rate': 0.05,
    'max_depth': 6,
    'n_estimators': 500,
    'early_stopping_rounds': 20,
    'seed': 42,
    'verbosity': 1 
}

model_params = {k:v for k,v in params.items() if k != 'early_stopping_rounds'}
model = xgb.XGBRegressor(**model_params)

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False 
    )
except Exception as e:
    print(f"Error during fitting: {e}")


# ---------------------------------------------------------
# 4. FINAL EVALUATION & LOGGING
# ---------------------------------------------------------
y_pred = model.predict(X_val)
y_train_pred=model.predict(X_train)

if 'IsHoliday' in X_val.columns:
    is_holiday = X_val['IsHoliday'].values
else:
    is_holiday = np.zeros(len(y_val))
if 'IsHoliday' in X_train.columns:
    is_holiday_train = X_train['IsHoliday'].values
else:
    is_holiday_train = np.zeros(len(y_train))

weights = np.where(is_holiday, 5, 1)
weights_train = np.where(is_holiday_train, 5, 1)

final_wmae = np.average(np.abs(y_val - y_pred), weights=weights)
final_mae = mean_absolute_error(y_val, y_pred)

train_wmae = np.average(np.abs(y_train - y_train_pred), weights=weights_train)
train_mae = mean_absolute_error(y_train, y_train_pred)
if run:
    # Log Final Metrics
    wandb.log({"final_validation_wmae": final_wmae, "final_validation_mae": final_mae})
    wandb.log({"train_wmae": train_wmae, "train_mae": train_mae})

    
    # --- FIXED FEATURE IMPORTANCE LOGGING ---
    importance = model.feature_importances_
    feature_names = X_train.columns.tolist()
    importance_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    importance_df = importance_df.sort_values(by='importance', ascending=False)
    
    # Use wandb.Table which is compatible with all recent versions
    table = wandb.Table(dataframe=importance_df.head(20))
    wandb.log({"feature_importance_table": table})
    
    # Alternatively, you can log a simple bar plot using matplotlib if you prefer visuals
    import matplotlib.pyplot as plt
    plt.figure(figsize=(10, 6))
    plt.barh(importance_df.head(20)['feature'], importance_df.head(20)['importance'])
    plt.gca().invert_yaxis()
    plt.title("Top 20 Feature Importances")
    plt.tight_layout()
    wandb.log({"feature_importance_plot": wandb.Image(plt)})
    plt.close()
    # ----------------------------------------
    
    # Save Model Artifact to W&B
    artifact = wandb.Artifact("xgboost-model-final", type="model")
    joblib.dump(model, "model.joblib")
    artifact.add_file("model.joblib")
    run.log_artifact(artifact)

print(f"Final Train WMAE: {train_wmae:.2f}")
print(f"Final Train MAE: {train_mae:.2f}")

print(f"Final Validation WMAE: {final_wmae:.2f}")
print(f"Final Validation MAE: {final_mae:.2f}")

if run:
    wandb.finish()


Final Train WMAE: 1626.19
Final Train MAE: 1479.13
Final Validation WMAE: 1331.99
Final Validation MAE: 1233.77


wandb: WARNING Artifact "xgboost-model-final" already exists with the same content. No new version will be created.


final_validation_mae,▁
final_validation_wmae,▁
train_mae,▁
train_wmae,▁
final_validation_mae,1233.76872
final_validation_wmae,1331.98917
train_mae,1479.12766
train_wmae,1626.19283


In [ ]:
print(f"Holiday Weeks in Train: {X_train['IsHoliday'].sum()}")
print(f"Holiday Weeks in Val:   {X_val['IsHoliday'].sum()}")
